# Food Nutrition Analyzer

## Problem Statement

In this project our task is to build a machine learning algorithm that gives nutrition information about a food item or a dish given an image of the food.

This is actually a two part problem:
  1. **What food is this?** - this is an image classifcation problem which could be solved with deep neural networks
  2. **How much of it is there and what's in it?** - this is a much harder portion size / nutrition etimation problem.

For the image classifer we will use transfer learning with ResNet50 - a pretrained deep convolutional neural network trained on the ImageNet dataset. We will use the learned weights of this model as a fixed feature extractor and build a simple feed-forward neural network to classify the images into one of 101 classes. Using a pretrained convolutional neural netowork is much more pratical than training one form scratch since doing so requires large amount of diverse data, serious computational resourses and time.

Most solutions for the nutition estimation problem found
online "fake" it by looking up the average nutrition per dish with an API call to the USDA FoodData Central API or a similar one. This is a reasonable place to start and we will use this approach to build a simple prototype. Another approach is to build a multi-task deep neural network such as the one proposed in Google Reseach's official paper for the [Nutrition 5K](https://arxiv.org/abs/2103.03375) dataset

## Data

This project uses the [Food-101](https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/) dataset which contains 101 000 food images with 101 different classes. For each class there are 250 test images and as well as 750 training images.

## Evaluation

The project will conclude with the evaluation of the machine learning model selected with a test data set.
The output of the predictions can be checked through metrics such as accuracy, precision, recall and F1 score.

## Import Python Modules

In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision.datasets import Food101
from torchvision.transforms import ToTensor
from collections import Counter
from PIL import Image

In [2]:
# drive.mount("/content/drive")

## Load Dataset

In [3]:
train_data = Food101(root="../data/food-101/", split="train", transform=ToTensor(), download=True)
test_data = Food101(root="../data/food-101/", split="test", transform=ToTensor(), download=True)

100%|██████████| 5.00G/5.00G [19:26<00:00, 4.28MB/s]  


## Exploratory Data Analysis

### Shape and Class Labels Distibution

The shape of the dataset is as expected. The train set contains 75750 images- 750 images per each class and the test set contains 25250 images- 250 images per each class.

In [ ]:
print(f"Train samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

All of the food classes in the dataset are perfectly balanced

In [ ]:
classes = train_data.classes

print(len(classes))
print(classes[:20])

In [ ]:
labels = train_data._labels
counts = Counter(labels)

print(counts)

Some of the images are noisier than other and have different contrast. They are also taken form different angles and some have wrong labels. All of these factors could make it dfficult for our model to differentiate them.

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(12, 9))

np.random.seed(42)

for ax in axes.ravel():
    random_idx = np.random.randint(0, len(test_data),)
    img, label = test_data[random_idx]

    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(classes[label])
    ax.axis('off')

plt.tight_layout()
plt.show()

### Visualization of Per-Class Samples

In [ ]:
target = classes.index("pizza")

indices = [i for i, y in enumerate(test_data._labels) if y == target]

fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for ax, idx in zip(axes.ravel(), indices[:10]):
    img, _ = test_data[idx]
    ax.imshow(img.permute(1, 2, 0))
    ax.axis("off")

plt.tight_layout()
plt.show()

### Image Size Analysis

Not all of the images are of the same size. Most of them are 512*512 but some are much smaller. This could make some of the images harder to classify by our model

In [ ]:
widths = []
heights = []

for img, _ in test_data:
    widths.append(img.shape[2])
    heights.append(img.shape[1])

print(widths[:10])
print(heights[:10])

In [ ]:
plt.hist(widths, bins=30)
plt.title("Image Widths")
plt.xlabel("Width")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.hist(heights, bins=30)
plt.title("Image Heights")
plt.xlabel("Height")
plt.ylabel("Count")
plt.show()

In [ ]:
print(min(widths), max(widths))
print(min(heights), max(heights))